In [18]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output

np.set_printoptions(precision=4, suppress=True)

In [19]:
# =========================================================
# FILES
# =========================================================
INPUT_CSV   = "input_min_conso.csv"
NEURON_CSV  = "neuron.csv"
SYN_EXC_CSV = "excitation.csv"
SYN_INH_CSV = "inhibition.csv"

# =========================================================
# HYPERPARAMETERS
# =========================================================
SEED             = 42
RHO              = 0.55
N_TRAIN_UNIFORM  = 700
N_TRAIN_BOUNDARY = 700
N_TEST           = 500

# ---- architecture ----
USE_SYMMETRIC_INPUTS = False
HIDDEN_SIZES         = [6, 6, 6]

# ---- search ----
N_RESTARTS       = 5
MAX_PASSES       = 8
MARGIN_SCALE     = 5e-6
FREQ_MERGE_TOL   = 1000.0

# ---- hardware search options ----
ALLOW_ZERO_SYNAPSES            = True
ZERO_SYNAPSE_INIT_PROB         = 0.20
USE_SMALLER_WIDTHS_DEEP_LAYERS = True
DEEP_EXC_MAX                   = 0.7717
DEEP_INH_MAX                   = 0.5194

# ---- score penalty ----
IMBALANCE_PENALTY_SCALE = 0.15

# ---- decision map only ----
LOGIT_SCALE_HZ_SAFE = 2e4

In [20]:
def notebook_progress(current, total, prefix="", suffix="", bar_len=30):
    filled = int(bar_len * current / total)
    bar = "█" * filled + "░" * (bar_len - filled)
    pct = int(100 * current / total)
    print(f"{prefix}[{bar}] {current}/{total} {pct}%  {suffix}")

In [21]:
def read_cadence_rows(csv_path):
    df = pd.read_csv(csv_path)

    if "Parameter" not in df.columns:
        raise ValueError(f"{csv_path}: missing 'Parameter' column")

    data_cols = [c for c in df.columns if c not in ["Parameter", "Unnamed: 1"]]

    def get_row(row_name, allow_non_numeric=False):
        row = df.loc[df["Parameter"] == row_name, data_cols]
        if row.empty:
            raise ValueError(f"Row '{row_name}' not found in {csv_path}")

        values = pd.to_numeric(row.iloc[0], errors="coerce").to_numpy(dtype=float)

        if (not allow_non_numeric) and np.isnan(values).any():
            bad_idx = np.where(np.isnan(values))[0]
            raise ValueError(
                f"Row '{row_name}' in {csv_path} contains non-numeric values at {bad_idx[:10]}"
            )
        return values

    return get_row

def get_clean_rows(reader, row_names):
    rows    = [reader(name, allow_non_numeric=True) for name in row_names]
    stacked = np.vstack(rows)
    mask    = np.all(np.isfinite(stacked), axis=0)
    return [r[mask] for r in rows], mask

In [22]:
def interp_clip(x, xp, fp):
    return np.interp(np.asarray(x, dtype=float), xp, fp, left=fp[0], right=fp[-1])

def interp2d_bilinear_vec(x, y, x_axis, y_axis, z_grid):
    x  = np.clip(np.asarray(x, dtype=float), x_axis[0], x_axis[-1])
    y  = np.clip(np.asarray(y, dtype=float), y_axis[0], y_axis[-1])

    ix = np.clip(np.searchsorted(x_axis, x, side="right") - 1, 0, len(x_axis) - 2)
    iy = np.clip(np.searchsorted(y_axis, y, side="right") - 1, 0, len(y_axis) - 2)

    x1 = x_axis[ix]
    x2 = x_axis[ix + 1]
    y1 = y_axis[iy]
    y2 = y_axis[iy + 1]

    tx = np.where(x2 == x1, 0.0, (x - x1) / (x2 - x1))
    ty = np.where(y2 == y1, 0.0, (y - y1) / (y2 - y1))

    return (
        z_grid[ix,   iy  ] * (1 - tx) * (1 - ty) +
        z_grid[ix+1, iy  ] * tx       * (1 - ty) +
        z_grid[ix,   iy+1] * (1 - tx) * ty       +
        z_grid[ix+1, iy+1] * tx       * ty
    )

def strictly_monotonic_branch(x, y):
    im    = int(np.argmax(y))
    x_m   = np.asarray(x[:im+1], dtype=float)
    y_m   = np.asarray(y[:im+1], dtype=float)
    order = np.argsort(x_m)
    return x_m[order], y_m[order]

def merge_close_values(values, tol):
    vals   = np.sort(np.asarray(values, dtype=float))
    groups = [[vals[0]]]
    for v in vals[1:]:
        if abs(v - np.mean(groups[-1])) <= tol:
            groups[-1].append(v)
        else:
            groups.append([v])
    return np.array([np.mean(g) for g in groups], dtype=float)

def snap_to_axis(values, axis):
    return np.array([axis[np.argmin(np.abs(axis - v))]
                     for v in np.asarray(values, dtype=float)])

def build_grid_tolerant(W_all, F_all, Y_all, freq_tol=1000.0):
    W_unique = np.unique(W_all)
    F_axis   = merge_close_values(np.unique(F_all), tol=freq_tol)
    F_snap   = snap_to_axis(F_all, F_axis)
    z_grid   = np.empty((len(W_unique), len(F_axis)))

    for i, w in enumerate(W_unique):
        for j, f in enumerate(F_axis):
            mask = np.isclose(W_all, w) & np.isclose(F_snap, f)
            vals = Y_all[mask]
            if len(vals) == 0:
                raise ValueError(f"Missing point for W={w}, Fbin={f}")
            z_grid[i, j] = np.mean(vals)

    return W_unique, F_axis, z_grid

In [23]:
inp = read_cadence_rows(INPUT_CSV)
INPUT_V_AXIS, INPUT_F_AXIS = strictly_monotonic_branch(inp("Vex"), inp("Fspikes"))

neuro = read_cadence_rows(NEURON_CSV)
NEURON_I_AXIS, NEURON_F_AXIS = strictly_monotonic_branch(neuro("Iex"), neuro("Fspikes"))

syn_exc = read_cadence_rows(SYN_EXC_CSV)
(W2_EXC, FPRE_EXC, IPOST_EXC), _ = get_clean_rows(syn_exc, ["W2", "Fpre", "Ipost"])
W_EXC_AXIS, F_EXC_AXIS, IPOST_EXC_GRID = build_grid_tolerant(
    W2_EXC, FPRE_EXC, IPOST_EXC, freq_tol=FREQ_MERGE_TOL
)

syn_inh = read_cadence_rows(SYN_INH_CSV)
(W2_INH, FPRE_INH, IPOST_INH), _ = get_clean_rows(syn_inh, ["W2", "Fpre", "Ipost"])
W_INH_AXIS, F_INH_AXIS, IPOST_INH_GRID = build_grid_tolerant(
    W2_INH, FPRE_INH, IPOST_INH, freq_tol=FREQ_MERGE_TOL
)

V_MIN = float(INPUT_V_AXIS[0])
V_MAX = float(INPUT_V_AXIS[-1])

INPUT_DIM   = 4 if USE_SYMMETRIC_INPUTS else 2
LAYER_SIZES = [INPUT_DIM] + HIDDEN_SIZES + [2]

In [24]:
def input_f_of_v(Vin_V):
    return interp_clip(Vin_V, INPUT_V_AXIS, INPUT_F_AXIS)

def neuron_f_of_i(Iin_A):
    return interp_clip(Iin_A, NEURON_I_AXIS, NEURON_F_AXIS)

In [25]:
def get_layer_candidates(layer_idx, n_layers):
    exc_axis = W_EXC_AXIS
    inh_axis = W_INH_AXIS

    if USE_SMALLER_WIDTHS_DEEP_LAYERS and layer_idx >= 1:
        exc_axis = exc_axis[exc_axis <= DEEP_EXC_MAX]
        inh_axis = inh_axis[inh_axis <= DEEP_INH_MAX]

        if len(exc_axis) == 0:
            exc_axis = W_EXC_AXIS
        if len(inh_axis) == 0:
            inh_axis = W_INH_AXIS

    cand = []
    if ALLOW_ZERO_SYNAPSES:
        cand.append((0, 0.0))
    cand += [(+1, float(w)) for w in exc_axis]
    cand += [(-1, float(w)) for w in inh_axis]
    return cand

ALL_CANDIDATES_PER_LAYER = [
    get_layer_candidates(l, len(LAYER_SIZES)-1)
    for l in range(len(LAYER_SIZES)-1)
]

In [26]:
def layer_forward_batch(Fin_batch, sign_matrix, width_matrix):
    B, N = Fin_batch.shape
    M    = sign_matrix.shape[1]

    F = np.broadcast_to(Fin_batch[:, :, None],    (B, N, M))
    W = np.broadcast_to(width_matrix[None, :, :], (B, N, M))
    S = np.broadcast_to(sign_matrix[None, :, :],  (B, N, M))

    Iexc = interp2d_bilinear_vec(W, F, W_EXC_AXIS, F_EXC_AXIS, IPOST_EXC_GRID)
    Iinh = interp2d_bilinear_vec(W, F, W_INH_AXIS, F_INH_AXIS, IPOST_INH_GRID)

    Iconn = np.where(S == 1, Iexc,
             np.where(S == -1, -Iinh, 0.0))

    Itot = np.sum(Iconn, axis=1)
    return neuron_f_of_i(Itot)

def network_forward_batch(Vin_batch, sign_matrices, width_matrices):
    F = input_f_of_v(Vin_batch)
    for sign_mat, width_mat in zip(sign_matrices, width_matrices):
        F = layer_forward_batch(F, sign_mat, width_mat)
    return F

In [27]:
def evaluate(Vin_batch, y_true, sign_matrices, width_matrices):
    y_true = np.asarray(y_true, dtype=int)
    Fout   = network_forward_batch(Vin_batch, sign_matrices, width_matrices)
    preds  = np.argmax(Fout, axis=1)

    mask0 = (y_true == 0)
    mask1 = (y_true == 1)

    acc0 = float(np.mean(preds[mask0] == 0)) if mask0.any() else 0.0
    acc1 = float(np.mean(preds[mask1] == 1)) if mask1.any() else 0.0
    bal_acc = 0.5 * (acc0 + acc1)

    margin0    = float(np.mean(Fout[mask0, 0] - Fout[mask0, 1])) if mask0.any() else 0.0
    margin1    = float(np.mean(Fout[mask1, 1] - Fout[mask1, 0])) if mask1.any() else 0.0
    bal_margin = 0.5 * (margin0 + margin1)

    imbalance_penalty = IMBALANCE_PENALTY_SCALE * abs(acc0 - acc1)

    score = bal_acc + MARGIN_SCALE * bal_margin - imbalance_penalty
    return preds, Fout, bal_acc, bal_margin, score

In [28]:
def make_uniform_dataset(n, rho=RHO, seed=0):
    rng    = np.random.default_rng(seed)
    X      = rng.uniform(-1.0, 1.0, size=(n, 2)).astype(np.float32)
    labels = (X[:, 0]**2 + X[:, 1]**2 <= rho**2).astype(np.int32)
    return X, labels

def make_boundary_dataset(n, rho=RHO, sigma_r=0.08, seed=0):
    rng    = np.random.default_rng(seed)
    theta  = rng.uniform(0, 2*np.pi, n)
    r      = rho + rng.normal(0, sigma_r, n)
    X      = np.column_stack([r*np.cos(theta), r*np.sin(theta)]).astype(np.float32)
    mask   = np.all(np.abs(X) <= 1.0, axis=1)
    X      = X[mask]
    labels = (X[:, 0]**2 + X[:, 1]**2 <= rho**2).astype(np.int32)
    return X, labels

def xy_to_voltage(X):
    X = np.asarray(X, dtype=np.float32)

    if USE_SYMMETRIC_INPUTS:
        X_hw = np.column_stack([X[:, 0], -X[:, 0], X[:, 1], -X[:, 1]])
    else:
        X_hw = X

    return (V_MIN + (X_hw + 1.0) * (V_MAX - V_MIN) / 2.0).astype(np.float32)

def make_datasets():
    X_u, y_u = make_uniform_dataset(N_TRAIN_UNIFORM,   rho=RHO, seed=SEED)
    X_b, y_b = make_boundary_dataset(N_TRAIN_BOUNDARY, rho=RHO, seed=SEED+1)

    X_train = np.vstack([X_u, X_b])
    y_train = np.hstack([y_u, y_b])

    rng = np.random.default_rng(SEED)
    idx = np.arange(len(X_train))
    rng.shuffle(idx)
    X_train, y_train = X_train[idx], y_train[idx]

    X_test, y_test = make_uniform_dataset(N_TEST, rho=RHO, seed=SEED+2)

    V_train = xy_to_voltage(X_train)
    V_test  = xy_to_voltage(X_test)

    print(f"Train: {len(y_train)} | inside={y_train.sum()} ({y_train.mean()*100:.0f}%)  outside={(1-y_train).sum()} ({(1-y_train).mean()*100:.0f}%)")
    print(f"Test : {len(y_test)}  | inside={y_test.sum()} ({y_test.mean()*100:.0f}%)  outside={(1-y_test).sum()} ({(1-y_test).mean()*100:.0f}%)")

    return X_train, y_train, X_test, y_test, V_train, V_test

In [29]:
def random_network(layer_sizes, rng):
    signs, widths = [], []

    for layer_idx, (n_in, n_out) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:])):
        if ALLOW_ZERO_SYNAPSES:
            s = rng.choice(
                [-1, 0, 1],
                size=(n_in, n_out),
                p=[(1 - ZERO_SYNAPSE_INIT_PROB) / 2,
                   ZERO_SYNAPSE_INIT_PROB,
                   (1 - ZERO_SYNAPSE_INIT_PROB) / 2]
            )
        else:
            s = rng.choice([-1, 1], size=(n_in, n_out))

        w = np.zeros((n_in, n_out))
        candidates = ALL_CANDIDATES_PER_LAYER[layer_idx]
        exc_only = [c[1] for c in candidates if c[0] == 1]
        inh_only = [c[1] for c in candidates if c[0] == -1]

        for i in range(n_in):
            for j in range(n_out):
                if s[i, j] == 1:
                    w[i, j] = rng.choice(exc_only)
                elif s[i, j] == -1:
                    w[i, j] = rng.choice(inh_only)
                else:
                    w[i, j] = 0.0

        signs.append(s)
        widths.append(w)

    return signs, widths

def coordinate_search(V_train, y_train, V_test, y_test, layer_sizes,
                      n_restarts=N_RESTARTS, max_passes=MAX_PASSES, seed=SEED):
    rng = np.random.default_rng(seed)

    all_conns = [(l, i, j)
                 for l, (n_in, n_out) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:]))
                 for i in range(n_in)
                 for j in range(n_out)]
    n_conns = len(all_conns)

    best_signs_g  = None
    best_widths_g = None
    best_te_score = -1e99
    best_tr_score = -1e99
    all_history   = []

    for restart in range(n_restarts):
        print(f"\n{'='*65}")
        print(f"RESTART {restart+1}/{n_restarts}")

        signs, widths = random_network(layer_sizes, rng)
        _, _, tr_acc, tr_mrg, tr_score = evaluate(V_train, y_train, signs, widths)
        _, _, te_acc, te_mrg, te_score = evaluate(V_test,  y_test,  signs, widths)

        history = [{
            "pass": 0,
            "tr_acc": tr_acc,
            "tr_mrg": tr_mrg,
            "te_acc": te_acc,
            "te_mrg": te_mrg
        }]
        print(f"  Init | train_acc={tr_acc:.3f} margin={tr_mrg*1e-3:.1f}kHz | test_acc={te_acc:.3f} margin={te_mrg*1e-3:.1f}kHz")

        t0 = time.time()
        for pass_idx in range(max_passes):
            improved   = False
            conn_order = list(all_conns)
            rng.shuffle(conn_order)

            for conn_num, (l, i, j) in enumerate(conn_order):
                best_s = signs[l][i, j]
                best_w = widths[l][i, j]
                best_c = tr_score

                for cand_s, cand_w in ALL_CANDIDATES_PER_LAYER[l]:
                    if cand_s == signs[l][i, j] and np.isclose(cand_w, widths[l][i, j]):
                        continue

                    old_s, old_w    = signs[l][i, j], widths[l][i, j]
                    signs[l][i, j]  = cand_s
                    widths[l][i, j] = cand_w

                    _, _, _, _, c = evaluate(V_train, y_train, signs, widths)

                    if c > best_c:
                        best_c = c
                        best_s = cand_s
                        best_w = cand_w

                    signs[l][i, j]  = old_s
                    widths[l][i, j] = old_w

                if best_c > tr_score:
                    signs[l][i, j]  = best_s
                    widths[l][i, j] = best_w
                    _, _, tr_acc, tr_mrg, tr_score = evaluate(V_train, y_train, signs, widths)
                    improved = True

                if conn_num % 5 == 0 or conn_num == n_conns - 1:
                    clear_output(wait=True)
                    print(f"Restart {restart+1}/{n_restarts} | Pass {pass_idx+1}/{max_passes}")
                    notebook_progress(conn_num+1, n_conns,
                                      prefix="Searching ",
                                      suffix=f"tr_acc={tr_acc:.3f}  margin={tr_mrg*1e-3:.1f}kHz  elapsed={time.time()-t0:.0f}s")

            _, _, te_acc, te_mrg, te_score = evaluate(V_test, y_test, signs, widths)
            history.append({
                "pass": pass_idx+1,
                "tr_acc": tr_acc,
                "tr_mrg": tr_mrg,
                "te_acc": te_acc,
                "te_mrg": te_mrg
            })

            print(f"Pass {pass_idx+1} finished | train_acc={tr_acc:.3f} | test_acc={te_acc:.3f}")

            if not improved:
                print("Converged")
                break

        if (te_score > best_te_score or
                (np.isclose(te_score, best_te_score) and tr_score > best_tr_score)):
            best_signs_g  = [s.copy() for s in signs]
            best_widths_g = [w.copy() for w in widths]
            best_te_score = te_score
            best_tr_score = tr_score
            all_history   = history

    clear_output(wait=True)
    return best_signs_g, best_widths_g, all_history

In [30]:
def print_model_info():
    print("=== HARDWARE ===")
    print(f"  Input  V: {V_MIN*1e3:.0f} mV -> {V_MAX*1e3:.0f} mV  F: {INPUT_F_AXIS[0]*1e-3:.1f} -> {INPUT_F_AXIS[-1]*1e-3:.1f} kHz")
    print(f"  Neuron I: {NEURON_I_AXIS[0]*1e12:.1f} pA -> {NEURON_I_AXIS[-1]*1e12:.0f} pA  F: {NEURON_F_AXIS[0]*1e-3:.1f} -> {NEURON_F_AXIS[-1]*1e-3:.1f} kHz")
    print(f"  Exc widths (um): {W_EXC_AXIS}")
    print(f"  Inh widths (um): {W_INH_AXIS}")

def print_network(signs, widths):
    names = (["Input->H1"]
             + [f"H{k}->H{k+1}" for k in range(1, len(signs)-1)]
             + [f"H{len(signs)-1}->Output"])
    print("\n=== NETWORK ===")
    for k, (s, w, name) in enumerate(zip(signs, widths, names)):
        print(f"  Layer {k+1} [{name}] {s.shape[0]}->{s.shape[1]}  exc={int(np.sum(s==1))} inh={int(np.sum(s==-1))} off={int(np.sum(s==0))}")
        print(f"    signs:\n{s}")
        print(f"    widths (um):\n{w}")

def plot_dataset(X, y, rho=RHO, title="Dataset"):
    theta = np.linspace(0, 2*np.pi, 400)
    plt.figure(figsize=(6, 6))
    plt.scatter(X[y==0, 0], X[y==0, 1], s=5, alpha=0.5, label="Outside")
    plt.scatter(X[y==1, 0], X[y==1, 1], s=5, alpha=0.5, label="Inside")
    plt.plot(rho*np.cos(theta), rho*np.sin(theta), "r-", lw=2, label="Boundary")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.gca().set_aspect("equal")
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_decision_map(signs, widths, rho=RHO, resolution=120):
    xs     = np.linspace(-1, 1, resolution)
    ys     = np.linspace(-1, 1, resolution)
    XX, YY = np.meshgrid(xs, ys)
    grid_X = np.column_stack([XX.ravel(), YY.ravel()])
    grid_V = xy_to_voltage(grid_X)

    Fout   = network_forward_batch(grid_V, signs, widths)
    pred   = np.argmax(Fout, axis=1).reshape(XX.shape)
    margin = (Fout[:, 1] - Fout[:, 0]).reshape(XX.shape) * 1e-3

    theta  = np.linspace(0, 2*np.pi, 400)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.contourf(XX, YY, pred, levels=[-0.5, 0.5, 1.5],
                colors=["#f4a3a3", "#a3d9a3"], alpha=0.7)
    ax.contour(XX, YY, margin, levels=[0], colors="k", linewidths=2)
    ax.plot(rho*np.cos(theta), rho*np.sin(theta), "r--", lw=2, label="True boundary")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Hardware decision")
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    ax   = axes[1]
    vmax = max(abs(float(margin.min())), abs(float(margin.max())), 1.0)
    cf   = ax.contourf(XX, YY, margin, levels=50, cmap="RdYlGn", vmin=-vmax, vmax=vmax)
    plt.colorbar(cf, ax=ax, label="F_out1 - F_out0  (kHz)")
    ax.contour(XX, YY, margin, levels=[0], colors="k", linewidths=2)
    ax.plot(rho*np.cos(theta), rho*np.sin(theta), "r--", lw=2, label="True boundary")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Frequency margin")
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.suptitle("Hardware network decision = argmax(F_out)", fontsize=11)
    plt.tight_layout()
    plt.show()

def plot_history(history):
    if not history:
        return
    passes  = [h["pass"]   for h in history]
    tr_accs = [h["tr_acc"] for h in history]
    te_accs = [h["te_acc"] for h in history]
    plt.figure(figsize=(7, 4))
    plt.plot(passes, tr_accs, "o-", label="Train balanced acc")
    plt.plot(passes, te_accs, "s-", label="Test balanced acc")
    plt.xlabel("Pass")
    plt.ylabel("Balanced accuracy")
    plt.title("Coordinate search — learning curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def print_results(label, Vin_batch, y_true, signs, widths):
    preds, Fout, bal_acc, bal_margin, score = evaluate(Vin_batch, y_true, signs, widths)
    y_true = np.asarray(y_true, dtype=int)
    mask0  = (y_true == 0)
    mask1  = (y_true == 1)
    acc0   = float(np.mean(preds[mask0] == 0)) if mask0.any() else 0.0
    acc1   = float(np.mean(preds[mask1] == 1)) if mask1.any() else 0.0
    print(f"\n=== {label} ===")
    print(f"  acc class 0 (outside) = {acc0:.4f}")
    print(f"  acc class 1 (inside)  = {acc1:.4f}")
    print(f"  balanced accuracy     = {bal_acc:.4f}")
    print(f"  balanced margin       = {bal_margin*1e-3:.2f} kHz")
    print(f"  score                 = {score:.6f}")

def demo_small_forward():
    Vin_vec_V = np.array([20e-3, 50e-3])

    sign_matrix = np.array([
        [ 1, -1],
        [-1,  1],
    ])

    width_matrix_um = np.array([
        [1.09,   0.325],
        [0.455,  1.410],
    ])

    Fin_vec_Hz = input_f_of_v(Vin_vec_V)
    F = Fin_vec_Hz.reshape(1, -1)
    Fout = layer_forward_batch(F, sign_matrix, width_matrix_um).reshape(-1)

    print("\n=== QUICK 2->2 TEST ===")
    print("Vin_vec (mV) =")
    print(Vin_vec_V * 1e3)
    print("Fin_vec (kHz) =")
    print(Fin_vec_Hz * 1e-3)
    print("Fout_vec (kHz) =")
    print(Fout * 1e-3)

In [ ]:
print_model_info()

print("\n=== ARCHITECTURE SETTINGS ===")
print(f"  USE_SYMMETRIC_INPUTS         = {USE_SYMMETRIC_INPUTS}")
print(f"  LAYER_SIZES                  = {LAYER_SIZES}")
print(f"  ALLOW_ZERO_SYNAPSES          = {ALLOW_ZERO_SYNAPSES}")
print(f"  USE_SMALLER_WIDTHS_DEEP      = {USE_SMALLER_WIDTHS_DEEP_LAYERS}")
print(f"  N_RESTARTS                   = {N_RESTARTS}")
print(f"  MAX_PASSES                   = {MAX_PASSES}")
print(f"  MARGIN_SCALE                 = {MARGIN_SCALE}")
print(f"  IMBALANCE_PENALTY_SCALE      = {IMBALANCE_PENALTY_SCALE}")
print(f"  DEEP_EXC_MAX                 = {DEEP_EXC_MAX}")
print(f"  DEEP_INH_MAX                 = {DEEP_INH_MAX}")

demo_small_forward()

print("\n=== DATASET ===")
X_train, y_train, X_test, y_test, V_train, V_test = make_datasets()

plot_dataset(X_train, y_train, title="Train (uniform + boundary)")
plot_dataset(X_test,  y_test,  title="Test (uniform)")

print("\n=== COORDINATE SEARCH ===")
print(f"Network    : {LAYER_SIZES}")
print(f"Connections: {sum(a*b for a,b in zip(LAYER_SIZES[:-1], LAYER_SIZES[1:]))}")
for l, cand in enumerate(ALL_CANDIDATES_PER_LAYER):
    print(f"Layer {l+1} candidates: {len(cand)}")

best_signs, best_widths, history = coordinate_search(
    V_train, y_train, V_test, y_test,
    layer_sizes=LAYER_SIZES,
    n_restarts=N_RESTARTS,
    max_passes=MAX_PASSES,
    seed=SEED,
)

print_network(best_signs, best_widths)
print_results("TRAIN", V_train, y_train, best_signs, best_widths)
print_results("TEST",  V_test,  y_test,  best_signs, best_widths)

plot_history(history)
plot_decision_map(best_signs, best_widths, rho=RHO)

Restart 2/5 | Pass 1/8
Searching [███░░░░░░░░░░░░░░░░░░░░░░░░░░░] 11/96 11%  tr_acc=0.500  margin=0.2kHz  elapsed=1s
